# INFO 4220/6220 — Assignment 2: single vs. multiple tie-breaking

This notebook re-implements the school-choice simulation from the R script in
pure Python (the original notebook imported `matchingMarkets`, which is an R
package and cannot be imported from Python).

Sixty students are assigned to four schools with capacities 5, 40, 15, and 10.
Student preferences are random permutations weighted toward school 1. School
priorities are fixed groups; ties are broken either by **one lottery for all
schools** (single tie-breaking, STB) or by **a separate lottery per school**
(multiple tie-breaking, MTB). The mechanism is student-proposing deferred
acceptance, so it is strategy-proof for students.

Run the cells top to bottom; the final cell prints the average number of
students assigned to their 1st–4th choice under each rule.


## Setup

Constants and the fixed priority groups.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

NUM_STUDENTS = 60
NUM_SCHOOLS = 4
CAPACITIES = (5, 40, 15, 10)
PREFERENCE_WEIGHTS = (5, 1, 2, 3)   # school 1 is the most popular first choice
NUM_SIMULATIONS = 5_000
SEED = 42

# Priority groups per school, highest priority first (0-based student IDs).
PRIORITY_GROUPS = {
    0: (range(0, 15), range(15, 30), range(30, 45), range(45, 60)),
    1: (range(45, 60), range(0, 15), range(15, 30), range(30, 45)),
    2: (range(30, 45), range(45, 60), range(0, 15), range(15, 30)),
    3: (range(15, 30), range(30, 45), range(45, 60), range(0, 15)),
}


## Deferred acceptance and tie-breaking

In [ ]:
def student_preferences(rng: np.random.Generator) -> np.ndarray:
    """Draw a strict preference order for every student."""
    weights = np.asarray(PREFERENCE_WEIGHTS, dtype=float)
    weights /= weights.sum()
    prefs = np.empty((NUM_STUDENTS, NUM_SCHOOLS), dtype=int)
    for student in range(NUM_STUDENTS):
        prefs[student] = rng.choice(NUM_SCHOOLS, size=NUM_SCHOOLS, replace=False, p=weights)
    return prefs


def strict_priorities(lottery_rank: np.ndarray, single: bool) -> list[dict[int, int]]:
    """Turn the priority groups into a strict ranking for each school.

    ``lottery_rank`` is either a 1-D array (one lottery, used by every school)
    or a 2-D array with one column per school (separate lotteries).
    """
    priorities = []
    for school in range(NUM_SCHOOLS):
        order = []
        for group in PRIORITY_GROUPS[school]:
            if single:
                order.extend(sorted(group, key=lambda student: lottery_rank[student]))
            else:
                order.extend(sorted(group, key=lambda student: lottery_rank[student, school]))
        priorities.append({student: rank for rank, student in enumerate(order)})
    return priorities


def deferred_acceptance(
    prefs: np.ndarray,
    priorities: list[dict[int, int]],
    capacities: tuple[int, ...],
) -> np.ndarray:
    """Student-proposing deferred acceptance; returns school per student (-1 if none)."""
    next_choice = np.zeros(NUM_STUDENTS, dtype=int)
    assignment = -np.ones(NUM_STUDENTS, dtype=int)
    held: list[list[int]] = [[] for _ in range(NUM_SCHOOLS)]
    free = list(range(NUM_STUDENTS))

    while free:
        student = free.pop()
        if next_choice[student] >= NUM_SCHOOLS:
            continue
        school = int(prefs[student, next_choice[student]])
        next_choice[student] += 1
        held[school].append(student)

        if len(held[school]) > capacities[school]:
            # Reject the lowest-priority student currently held by the school.
            worst = max(held[school], key=lambda candidate: priorities[school][candidate])
            held[school].remove(worst)
            assignment[worst] = -1
            free.append(worst)
            if worst != student:
                assignment[student] = school
        else:
            assignment[student] = school

    return assignment


def rank_counts(assignment: np.ndarray, prefs: np.ndarray) -> np.ndarray:
    """Count how many students received their 1st, 2nd, 3rd, or 4th choice."""
    counts = np.zeros(NUM_SCHOOLS, dtype=int)
    for student, school in enumerate(assignment):
        if school >= 0:
            counts[int(np.where(prefs[student] == school)[0][0])] += 1
    return counts


## Simulation

5,000 draws of student preferences, with both tie-breaking rules.

In [3]:
rng = np.random.default_rng(SEED)
single_totals = np.zeros(NUM_SCHOOLS, dtype=float)
multiple_totals = np.zeros(NUM_SCHOOLS, dtype=float)

for _ in range(NUM_SIMULATIONS):
    prefs = student_preferences(rng)

    single_lottery = rng.permutation(NUM_STUDENTS)
    single_rank = np.empty(NUM_STUDENTS, dtype=int)
    single_rank[single_lottery] = np.arange(NUM_STUDENTS)
    single_priorities = strict_priorities(single_rank, single=True)
    single_totals += rank_counts(deferred_acceptance(prefs, single_priorities, CAPACITIES), prefs)

    multiple_rank = np.empty((NUM_STUDENTS, NUM_SCHOOLS), dtype=int)
    for school in range(NUM_SCHOOLS):
        lottery = rng.permutation(NUM_STUDENTS)
        multiple_rank[lottery, school] = np.arange(NUM_STUDENTS)
    multiple_priorities = strict_priorities(multiple_rank, single=False)
    multiple_totals += rank_counts(deferred_acceptance(prefs, multiple_priorities, CAPACITIES), prefs)

single_avg = single_totals / NUM_SIMULATIONS
multiple_avg = multiple_totals / NUM_SIMULATIONS

print(f"{'Tie-breaking':<12}{'1st':>10}{'2nd':>10}{'3rd':>10}{'4th':>10}")
print("-" * 52)
for label, row in (("Single", single_avg), ("Multiple", multiple_avg)):
    print(f"{label:<12}" + "".join(f"{value:>10.2f}" for value in row))


Tie-breaking       1st       2nd       3rd       4th
----------------------------------------------------
Single           16.08     16.81     16.84     10.26
Multiple         16.06     16.84     16.88     10.23


## Reading the results

Both rules fill all 60 seats, so the comparison is about **which** choices
students receive, not how many are assigned. With only 5 seats at the most
popular school, the first-choice count is capped by capacity; the remaining
students are spread across their second through fourth choices. The averages
sum to 60 students per simulation.

The simulation is a teaching model, not a policy recommendation:

* There are only four schools and a fixed priority table.
* Preferences are independent random permutations, weighted toward school 1.
* The comparison uses the same student preferences within each simulation, which
  is the right way to isolate the effect of tie-breaking.
* Re-running with a different `SEED` changes the numbers slightly.

The R scripts in this folder (`INFO6220-A2.R` and
`INFO6220-A2-custom-DA.R`) implement the same comparison with the
`matchingMarkets` package and a hand-written DA function. The pure-Python
implementation here reproduces the R averages (about 16.1 first choices for
both rules), which is a useful cross-check of the mechanism.
